# Parte 2 · API de lluvias

**Pregunta del trabajo:** ¿el Estado declara emergencia donde más llueve?

En la Parte 1 contamos los decretos de emergencia. Aquí medimos la otra mitad:
**cuánto llovió en cada departamento** entre el 1 de enero y el 31 de mayo de **2022**.

No podemos medir la lluvia de un departamento entero, así que usamos su
**capital** como punto de referencia. El camino tiene cuatro pasos:

1. Armar la lista departamento → capital (Wikipedia).
2. Convertir cada capital en coordenadas (API de geocodificación de Open-Meteo).
3. Pedir la lluvia diaria de cada punto (API histórica de Open-Meteo).
4. Resumir por departamento y guardar.

In [1]:
import io
import os
import time

import pandas as pd
import requests

## 1. Departamentos y capitales desde Wikipedia

Wikipedia rechaza los pedidos que no se identifican como navegador (error 403).
Por eso primero descargamos la página con `requests` y un `User-Agent`, y luego
le pasamos el texto a `pd.read_html`.

In [2]:
url_wikipedia = "https://es.wikipedia.org/wiki/Departamentos_del_Per%C3%BA"

respuesta = requests.get(url_wikipedia, headers={"User-Agent": "Mozilla/5.0"}, timeout=30)
respuesta.raise_for_status()

tablas = pd.read_html(io.StringIO(respuesta.text))
len(tablas)

6

La página tiene varias tablas. Antes de elegir una, miramos qué columnas trae cada
una: así sabemos cuál es la de departamentos.

In [3]:
for numero, tabla in enumerate(tablas):
    print(numero, tabla.shape, list(tabla.columns)[:6])

0 (1, 1) ['Departamentos del Perú (con las provincias de régimen especial)']
1 (24, 11) ['Bandera', 'Escudo', 'Departamento', 'Nombre oficial', 'Abreviatura Nacional', 'Capital']
2 (8, 5) ['Departamento', 'Capital', 'Creación', 'Desaparición', 'Causa']
3 (2, 11) ['Bandera', 'Escudo', 'Provincia', 'Nombre oficial', 'Abreviatura Nacional', 'Capital']
4 (6, 4) ['Provincia', 'Capital', 'Creación', 'Desaparición']
5 (1, 2) [0, 1]


La tabla `1` es la de departamentos. Nos quedamos con dos columnas y les ponemos
los nombres que pide el trabajo.

In [4]:
capitales = tablas[1][["Departamento", "Capital"]]
capitales.columns = ["departamento", "capital"]
capitales

,departamento,capital
0,Amazonas,Chachapoyas
1,Áncash,Huaraz
2,Apurímac,Abancay
3,Arequipa,Arequipa
4,Ayacucho,Ayacucho
5,Cajamarca,Cajamarca
6,Cusco,Cusco
7,Huancavelica,Huancavelica
8,Huánuco,Huánuco
9,Ica,Ica


### Verificación: ¿están los 25?

Esperamos **25 filas** (24 departamentos y el Callao).

In [5]:
print("Filas:", len(capitales))
print("¿Está el Callao?", "Callao" in capitales["departamento"].values)

Filas: 24
¿Está el Callao? False


Falta el **Callao**. En esta página no aparece en la tabla de departamentos
porque es una *provincia constitucional*. Está en otra tabla de la misma página
(la `3`), la de **provincias de régimen especial**, junto con Lima Metropolitana.

In [6]:
tablas[3][["Provincia", "Capital"]]

,Provincia,Capital
0,Callao[nota 6],Callao
1,Lima[nota 7],Lima


Tomamos la fila del Callao de esa tabla. El nombre trae una nota de Wikipedia
pegada (`[nota 6]`), así que lo dejamos solo como `Callao` para que después
pueda cruzarse con las demás tablas.

In [7]:
fila_callao = tablas[3][tablas[3]["Provincia"].str.startswith("Callao")]
fila_callao = fila_callao[["Provincia", "Capital"]]
fila_callao.columns = ["departamento", "capital"]
fila_callao["departamento"] = "Callao"

capitales = pd.concat([capitales, fila_callao], ignore_index=True)
print("Filas:", len(capitales))

Filas: 25


### Verificación: la columna `capital`

Ya hay 25 filas, pero falta revisar la columna `capital`. Si una celda trae texto
extra, la API no va a encontrar la ciudad. Buscamos las que tienen paréntesis.

In [8]:
capitales[capitales["capital"].str.contains("(", regex=False)]

,departamento,capital
13,Lima,Huacho (de facto)


**Lima** aparece como *"Huacho (de facto)"*. Hay dos opciones: usar **Lima** (la
ciudad) o **Huacho**.

Elegimos **Huacho** porque es la sede del Gobierno Regional de Lima. Los decretos
de emergencia que mencionan el departamento de Lima suelen referirse a sus
provincias fuera de Lima Metropolitana (Huaura, Huarochirí, Canta, Cajatambo…),
y Lima Metropolitana tiene un régimen especial aparte.

Igual hay que tener en cuenta que Huacho y Lima están en la costa, donde casi no
llueve. En cambio, las emergencias del departamento suelen ocurrir en la sierra
de Lima. Es una **limitación** que vamos a mencionar en la Parte 3.

In [9]:
capitales.loc[capitales["departamento"] == "Lima", "capital"] = "Huacho"
capitales

,departamento,capital
0,Amazonas,Chachapoyas
1,Áncash,Huaraz
2,Apurímac,Abancay
3,Arequipa,Arequipa
4,Ayacucho,Ayacucho
5,Cajamarca,Cajamarca
6,Cusco,Cusco
7,Huancavelica,Huancavelica
8,Huánuco,Huánuco
9,Ica,Ica


## 2. De capital a coordenadas: API de geocodificación

La API de lluvia no entiende nombres de ciudades: necesita **latitud y longitud**.
Por eso primero le preguntamos a la API de geocodificación dónde queda cada capital.

Como en la sesión 5: **antes de recorrer las 25, miramos la respuesta de una sola**.

In [10]:
url_geocodificacion = "https://geocoding-api.open-meteo.com/v1/search"

respuesta = requests.get(
    url_geocodificacion,
    params={"name": "Chachapoyas", "count": 10, "language": "es"},
    timeout=30,
)
respuesta.raise_for_status()

for resultado in respuesta.json()["results"]:
    print(resultado["country_code"], "|", resultado.get("admin1"), "|",
          resultado["latitude"], resultado["longitude"])

PE | Departamento de Amazonas | -6.23169 -77.86903
AR | Provincia de Salta | -24.74735 -65.38527
PE | Ancash | -8.97357 -78.49754
PE | Departamento de Amazonas | -6.20181 -77.85606


La API devuelve **varias Chachapoyas**: una en Argentina e incluso **otra en el
Perú, pero en Áncash**. Entonces:

- No basta con tomar el primer resultado: hay que quedarse con el primero que sea
  del Perú (`country_code == "PE"`).
- Tampoco basta con que sea del Perú: después hay que **verificar** que `admin1`
  (el departamento que devuelve la API) sea el correcto.

Este pedido se va a repetir 25 veces, así que lo ponemos en una función. La
probamos con **Trujillo**, que también existe en Venezuela, Honduras y España.

In [11]:
def buscar_en_peru(capital):
    """Devuelve el primer resultado de la API que esté en el Perú."""
    respuesta = requests.get(
        url_geocodificacion,
        params={"name": capital, "count": 10, "language": "es"},
        timeout=30,
    )
    respuesta.raise_for_status()
    # Si la API no encuentra nada, la respuesta no trae la llave "results"
    resultados = respuesta.json().get("results", [])
    for resultado in resultados:
        if resultado["country_code"] == "PE":
            return resultado
    return None


buscar_en_peru("Trujillo")

{'id': 3691175,
 'name': 'Trujillo',
 'latitude': -8.11599,
 'longitude': -79.02998,
 'elevation': 31.0,
 'feature_code': 'PPLA',
 'country_code': 'PE',
 'admin1_id': 3695781,
 'admin2_id': 3691174,
 'admin3_id': 8351227,
 'timezone': 'America/Lima',
 'population': 919899,
 'country_id': 3932488,
 'country': 'Perú',
 'admin1': 'Departamento de La Libertad',
 'admin2': 'Provincia de Trujillo',
 'admin3': 'Provincia de Trujillo'}

Ahora recorremos las 25 capitales, con una pausa de 1 segundo entre pedidos para
no saturar la API.

In [12]:
latitudes = []
longitudes = []
departamentos_api = []

for capital in capitales["capital"]:
    resultado = buscar_en_peru(capital)
    if resultado is None:
        print("No se encontró en el Perú:", capital)
        latitudes.append(None)
        longitudes.append(None)
        departamentos_api.append(None)
    else:
        latitudes.append(resultado["latitude"])
        longitudes.append(resultado["longitude"])
        departamentos_api.append(resultado.get("admin1"))
    time.sleep(1)

capitales["admin1"] = departamentos_api
capitales["latitud"] = latitudes
capitales["longitud"] = longitudes

### Verificación: ¿todas las capitales están en su departamento?

`admin1` no viene escrito igual que nuestra lista: a veces dice *"Departamento de
Cusco"*, a veces *"Ancash"* sin tilde, y el Callao aparece como *"Provincia
Constitucional del Callao"*. Para compararlos, pasamos los dos textos a
minúsculas, les quitamos las tildes y revisamos si el nombre del departamento
**está dentro** de `admin1`.

In [13]:
def sin_tildes(texto):
    texto = texto.lower()
    for con_tilde, sin_tilde in [("á", "a"), ("é", "e"), ("í", "i"), ("ó", "o"), ("ú", "u")]:
        texto = texto.replace(con_tilde, sin_tilde)
    return texto


coincide = []
for departamento, admin1 in zip(capitales["departamento"], capitales["admin1"]):
    coincide.append(sin_tildes(departamento) in sin_tildes(admin1))

capitales["coincide"] = coincide
capitales[["departamento", "capital", "admin1", "latitud", "longitud", "coincide"]]

,departamento,capital,admin1,latitud,longitud,coincide
0,Amazonas,Chachapoyas,Departamento de Amazonas,-6.23169,-77.86903,True
1,Áncash,Huaraz,Ancash,-9.52614,-77.52869,True
2,Apurímac,Abancay,Apurímac,-13.63426,-72.88380,True
3,Arequipa,Arequipa,Departamento de Arequipa,-16.39899,-71.53747,True
4,Ayacucho,Ayacucho,Departamento de Ayacucho,-13.16380,-74.22345,True
5,Cajamarca,Cajamarca,Departamento de Cajamarca,-7.16378,-78.50027,True
6,Cusco,Cusco,Departamento de Cusco,-13.53188,-71.96701,True
7,Huancavelica,Huancavelica,Departamento de Huancavelica,-12.78691,-74.97298,True
8,Huánuco,Huánuco,Departamento de Huánuco,-9.92882,-76.23989,True
9,Ica,Ica,Departamento de Ica,-14.07538,-75.73422,True


In [14]:
print("Capitales en el departamento correcto:", capitales["coincide"].sum(), "de", len(capitales))

Capitales en el departamento correcto: 25 de 25


**Cómo elegimos el resultado correcto:** de los resultados que devuelve la API,
tomamos el primero con `country_code == "PE"`. La API ordena los resultados por
importancia, así que el primero del Perú suele ser la capital y no un pueblo
pequeño con el mismo nombre. Luego lo comprobamos fila por fila: las 25 tienen
`coincide = True`, o sea, ninguna capital cayó en otro país ni en otro departamento.

Que el orden haya funcionado esta vez no significa que siempre vaya a funcionar.
Por ejemplo, la segunda Chachapoyas del Perú está en Áncash: si la API cambiara
el orden, sin esta verificación no nos daríamos cuenta.

## 3. Lluvia diaria: API histórica de Open-Meteo

De nuevo, primero miramos la respuesta de **una sola capital** (Chachapoyas)
antes de recorrer todas.

In [15]:
url_historico = "https://archive-api.open-meteo.com/v1/archive"

parametros = {
    "latitude": capitales.loc[0, "latitud"],
    "longitude": capitales.loc[0, "longitud"],
    "start_date": "2022-01-01",
    "end_date": "2022-05-31",
    "daily": "precipitation_sum",
    "timezone": "America/Lima",
}

respuesta = requests.get(url_historico, params=parametros, timeout=30)
respuesta.raise_for_status()
datos = respuesta.json()
datos.keys()

dict_keys(['latitude', 'longitude', 'generationtime_ms', 'utc_offset_seconds', 'timezone', 'timezone_abbreviation', 'elevation', 'daily_units', 'daily'])

Lo que nos interesa está dentro de `daily`: una lista de fechas (`time`) y otra
con los milímetros de lluvia de cada día (`precipitation_sum`), en el mismo orden.

In [16]:
print("Días recibidos:", len(datos["daily"]["time"]))
print("Unidad:", datos["daily_units"]["precipitation_sum"])
print("Primeros 5 días:", datos["daily"]["precipitation_sum"][:5])

Días recibidos: 151
Unidad: mm
Primeros 5 días: [3.1, 3.8, 0.9, 2.5, 1.2]


Del 1 de enero al 31 de mayo de 2022 hay 31 + 28 + 31 + 30 + 31 = **151 días**.
Coincide.

Ahora lo repetimos para las 25 capitales. Juntamos todo en una tabla **larga**,
con una fila por departamento y por día. Así después podemos sumar y contar con
`groupby`, como en la sesión 3.

In [17]:
tablas_diarias = []

for indice, fila in capitales.iterrows():
    parametros["latitude"] = fila["latitud"]
    parametros["longitude"] = fila["longitud"]

    respuesta = requests.get(url_historico, params=parametros, timeout=30)
    respuesta.raise_for_status()
    diario = respuesta.json()["daily"]

    tabla = pd.DataFrame({
        "departamento": fila["departamento"],
        "fecha": diario["time"],
        "lluvia_mm": diario["precipitation_sum"],
    })
    tablas_diarias.append(tabla)
    time.sleep(1)

lluvia_diaria = pd.concat(tablas_diarias, ignore_index=True)
print("Filas:", len(lluvia_diaria), "| deberían ser 25 × 151 =", 25 * 151)
lluvia_diaria.head()

Filas: 3775 | deberían ser 25 × 151 = 3775


,departamento,fecha,lluvia_mm
0,Amazonas,2022-01-01,3.1
1,Amazonas,2022-01-02,3.8
2,Amazonas,2022-01-03,0.9
3,Amazonas,2022-01-04,2.5
4,Amazonas,2022-01-05,1.2


### Verificación: días sin dato

Si la API no tiene el dato de un día, manda `None`, que en pandas se vuelve `NaN`.
Hay que contarlos **antes** de sumar, porque `.sum()` los salta sin avisar: un
departamento con días sin dato parecería menos lluvioso de lo que fue.

In [18]:
print("Días sin dato:", lluvia_diaria["lluvia_mm"].isna().sum())

Días sin dato: 0


**Resultado:** no hay ningún día sin dato en las 25 capitales. No tuvimos que
corregir nada y las sumas usan los 151 días completos.

Si hubiera habido días sin dato, no los habríamos contado como 0. Habríamos
reportado cuántos faltan en cada departamento para tenerlo en cuenta al comparar.

## 4. Resumen por departamento

- `lluvia_total_mm`: la suma de la lluvia de los 151 días.
- `dias_lluvia_fuerte`: cuántos días tuvieron **20 mm o más**.

Para contar días, primero creamos una columna `True`/`False` y luego la sumamos:
cada `True` cuenta como 1.

In [19]:
lluvia_diaria["lluvia_fuerte"] = lluvia_diaria["lluvia_mm"] >= 20

resumen = lluvia_diaria.groupby("departamento").agg(
    lluvia_total_mm=("lluvia_mm", "sum"),
    dias_lluvia_fuerte=("lluvia_fuerte", "sum"),
).reset_index()

resumen["lluvia_total_mm"] = resumen["lluvia_total_mm"].round(1)
resumen.sort_values("lluvia_total_mm", ascending=False)

,departamento,lluvia_total_mm,dias_lluvia_fuerte
14,Loreto,1581.8,25
23,Ucayali,1288.8,20
24,Áncash,975.3,7
10,Junín,910.4,5
15,Madre de Dios,908.0,13
7,Huancavelica,803.8,1
20,San Martín,789.8,4
19,Puno,779.5,3
17,Pasco,774.0,0
4,Cajamarca,773.3,1


Juntamos el resumen con la tabla de capitales y coordenadas usando el nombre del
departamento. Aquí el nombre es seguro para cruzar porque las dos tablas salieron
de la misma lista. El cruce con los decretos, en cambio, se hace por **ubigeo** en
la Parte 3.

In [20]:
lluvias_por_departamento = capitales.merge(resumen, on="departamento", how="left")
lluvias_por_departamento = lluvias_por_departamento[
    ["departamento", "capital", "latitud", "longitud", "lluvia_total_mm", "dias_lluvia_fuerte"]
]
print("Filas:", len(lluvias_por_departamento))
lluvias_por_departamento

Filas: 25


,departamento,capital,latitud,longitud,lluvia_total_mm,dias_lluvia_fuerte
0,Amazonas,Chachapoyas,-6.23169,-77.86903,436.5,1
1,Áncash,Huaraz,-9.52614,-77.52869,975.3,7
2,Apurímac,Abancay,-13.63426,-72.88380,711.8,2
3,Arequipa,Arequipa,-16.39899,-71.53747,189.9,0
4,Ayacucho,Ayacucho,-13.16380,-74.22345,485.1,0
5,Cajamarca,Cajamarca,-7.16378,-78.50027,773.3,1
6,Cusco,Cusco,-13.53188,-71.96701,710.7,3
7,Huancavelica,Huancavelica,-12.78691,-74.97298,803.8,1
8,Huánuco,Huánuco,-9.92882,-76.23989,444.8,1
9,Ica,Ica,-14.07538,-75.73422,25.1,0


## 5. Guardar

Usamos `encoding="utf-8-sig"` para que las tildes (Áncash, Junín, San Martín) se
vean bien si alguien abre el archivo en Excel.

In [21]:
os.makedirs("datos", exist_ok=True)
lluvias_por_departamento.to_csv(
    "datos/lluvias_por_departamento.csv", index=False, encoding="utf-8-sig"
)
pd.read_csv("datos/lluvias_por_departamento.csv").shape

(25, 6)

---
## Lo que hicimos

| Paso | Código |
|---|---|
| Leer una tabla de Wikipedia | `requests.get(url, headers=...)` + `pd.read_html(io.StringIO(...))` |
| Completar el Callao | `pd.concat([capitales, fila_callao])` |
| Limpiar la capital de Lima | `capitales.loc[..., "capital"] = "Huacho"` |
| Pedir coordenadas | `requests.get(url_geocodificacion, params={"name": ...})` |
| Quedarse con el resultado del Perú | `if resultado["country_code"] == "PE"` |
| Verificar el departamento | `sin_tildes(departamento) in sin_tildes(admin1)` |
| Pedir la lluvia diaria | `requests.get(url_historico, params={...})` |
| No saturar la API | `time.sleep(1)` |
| Contar días sin dato | `.isna().sum()` |
| Total y días fuertes | `.groupby("departamento").agg(...)` |
| Guardar | `.to_csv(..., encoding="utf-8-sig")` |

### Limitaciones para la Parte 3

1. **Un punto por departamento.** La lluvia de la capital no representa a todo el
   departamento. Por ejemplo, Huacho (Lima) e Ica están en la costa, pero sus
   emergencias suelen ocurrir en la sierra.
2. **No es una estación meteorológica.** Open-Meteo estima la lluvia con un modelo
   (reanálisis) sobre una grilla, no con un pluviómetro en la ciudad.
3. **Lluvia total no es igual a daño.** Un departamento puede tener mucha lluvia
   repartida en muchos días sin desastres, y otro puede tener un solo día extremo
   que cause huaicos.